# Myopsia: Notebook 02 - Model Training, Benchmarking & Export
### Project: AI-Based Eye Disease Prediction and Clinical Decision Support System Using Non-Image Clinical Data

> **Academic & Clinical Disclaimer:** This notebook trains synthetic machine learning classifiers on simulated tabular records for college educational purposes.

---
### Objectives of this Notebook:
1. **Build an End-to-End Scikit-Learn Pipeline:** Missing value imputation, standard scaling, and one-hot encoding.
2. **Train Multiple Candidate Classifiers:**
   - Logistic Regression (Linear baseline)
   - Decision Tree (Interpretable tree)
   - Gradient Boosting (Boosting ensemble)
   - Random Forest (Champion bagging ensemble with balanced weights)
3. **Multi-Class Evaluation:** Compute Accuracy, Macro Precision, Macro Recall, and Macro F1.
4. **Diagnostic Confusion Matrix:** Analyze classification boundaries and clinical overlaps.
5. **Feature Importance Analysis:** Uncover the clinical drivers behind the predictions.
6. **Model Serialization:** Export pipeline to `.joblib` for web deployment.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 110

print("Imported model training dependencies.")


## 1. Load Data & Define Feature Partitions

In [ ]:
df = pd.read_csv('../eye_disease_dataset.csv')
X = df.drop(columns=['patient_id', 'eye_disease'])
y = df['eye_disease']

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Numerical features: {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")


## 2. Preprocessing & Stratified Train/Test Split

In [ ]:
# Numerical Transformer: Median imputation + Standard Scaling
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Transformer: Most frequent imputation + One-Hot Encoding
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# 80/20 Stratified Partition
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training samples: {len(X_train)} | Testing samples: {len(X_test)}")


## 3. Train Candidate Classifiers & Benchmark Comparisons

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=14, class_weight='balanced', random_state=42)
}

results = []
pipelines = {}

for name, clf in models.items():
    pipe = Pipeline([
        ('prep', preprocessor),
        ('clf', clf)
    ])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe
    
    y_pred = pipe.predict(X_test)
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred) * 100, 2),
        'Macro F1': round(f1_score(y_test, y_pred, average='macro') * 100, 2)
    })

comparison_df = pd.DataFrame(results).set_index('Model')
print("Model Benchmark Summary (%):")
print(comparison_df)


## 4. Benchmark Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
comparison_df.plot(kind='bar', ax=ax, colormap='Accent', width=0.6)
ax.set_title('Model Performance Comparison on Holdout Test Set', fontsize=12, fontweight='bold')
ax.set_ylabel('Score (%)')
ax.set_ylim(85, 100)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.3),
                ha='center', fontsize=9)
plt.tight_layout()
plt.show()


## 5. Champion Random Forest Detailed Evaluation & Confusion Matrix

In [ ]:
champion_pipe = pipelines['Random Forest']
y_pred_champ = champion_pipe.predict(X_test)

print("Classification Report (Champion Random Forest):")
print(classification_report(y_test, y_pred_champ))

cm = confusion_matrix(y_test, y_pred_champ, labels=champion_pipe.classes_)
plt.figure(figsize=(8.5, 6.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=champion_pipe.classes_, yticklabels=champion_pipe.classes_)
plt.title('Random Forest Holdout Confusion Matrix (N = 1,000)', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Diagnosis')
plt.ylabel('True Diagnosis')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


## 6. Gini Feature Importance Analysis

In [ ]:
rf_clf = champion_pipe.named_steps['clf']
prep = champion_pipe.named_steps['prep']

cat_encoder = prep.named_transformers_['cat'].named_steps['onehot']
cat_names = cat_encoder.get_feature_names_out(cat_cols).tolist()
all_features = num_cols + cat_names

importances = pd.Series(rf_clf.feature_importances_, index=all_features).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(15).iloc[::-1].plot(kind='barh', color='#2b5c8f')
plt.title('Top 15 Most Predictive Clinical Features in Random Forest', fontsize=12, fontweight='bold')
plt.xlabel('Gini Importance')
plt.tight_layout()
plt.show()


## 7. Model Serialization & Export for Web Serving

In [ ]:
os.makedirs('../saved_models', exist_ok=True)
export_path = '../saved_models/eye_disease_pipeline.joblib'

joblib.dump(champion_pipe, export_path, compress=3)
print(f"Exported end-to-end inference pipeline to: {export_path}")

# Test reload and sample inference
loaded_pipe = joblib.load(export_path)
test_sample = X_test.iloc[[0]]
print("Sample prediction:", loaded_pipe.predict(test_sample)[0])
print("Sample probabilities:", loaded_pipe.predict_proba(test_sample)[0].round(3))
